In [1]:
import pandas as pd
import numpy as np
import re

from scipy.interpolate import interp1d
from scipy.interpolate import PchipInterpolator

from sklearn.metrics import mean_squared_error

In [2]:
df = pd.read_csv("../data/raw/dataset.csv")

ce_cols = [c for c in df.columns if c.endswith("CE")]
pe_cols = [c for c in df.columns if c.endswith("PE")]

In [3]:
def get_strike(col):
    return int(
        re.search(
            r'JAN26(\d+)(CE|PE)$',
            col
        ).group(1)
    )

ce_cols = sorted(
    ce_cols,
    key=get_strike
)

pe_cols = sorted(
    pe_cols,
    key=get_strike
)

ce_strikes = np.array(
    [get_strike(c) for c in ce_cols]
)

pe_strikes = np.array(
    [get_strike(c) for c in pe_cols]
)

In [4]:
def linear_fill(values, strikes):

    values = values.copy()

    mask = ~np.isnan(values)

    if mask.sum() < 2:
        return values

    f = interp1d(
        strikes[mask],
        values[mask],
        kind="linear",
        fill_value="extrapolate",
        bounds_error=False
    )

    values[~mask] = f(
        strikes[~mask]
    )

    return values

In [5]:
def pchip_fill(values, strikes):

    values = values.copy()

    mask = ~np.isnan(values)

    if mask.sum() < 2:
        return values

    f = PchipInterpolator(
        strikes[mask],
        values[mask],
        extrapolate=True
    )

    values[~mask] = f(
        strikes[~mask]
    )

    return values

In [6]:
def gap_aware_row(row, cols, strikes):

    values = row[cols].astype(float).values.copy()

    result = values.copy()

    n = len(values)

    i = 0

    while i < n:

        if np.isnan(values[i]):

            start = i

            while i < n and np.isnan(values[i]):
                i += 1

            end = i

            gap_size = end - start

            temp = values.copy()

            if gap_size == 1:

                temp = linear_fill(
                    temp,
                    strikes
                )

            else:

                temp = pchip_fill(
                    temp,
                    strikes
                )

            result[start:end] = temp[
                start:end
            ]

        else:

            i += 1

    return result

In [10]:
def gap_atm_row(row, cols, strikes):

    values = row[cols].astype(float).values.copy()

    result = values.copy()

    # Full ATM-aware fill
    atm_result = atm_aware_row(
        row,
        cols,
        strikes
    )

    # Full linear fill
    linear_result = linear_fill(
        values,
        strikes
    )

    n = len(values)
    i = 0

    while i < n:

        if np.isnan(values[i]):

            start = i

            while i < n and np.isnan(values[i]):
                i += 1

            end = i

            gap_size = end - start

            if gap_size == 1:

                result[start:end] = linear_result[
                    start:end
                ]

            else:

                result[start:end] = atm_result[
                    start:end
                ]

        else:

            i += 1

    return result

In [7]:
def evaluate_gap_aware(seed=42):

    np.random.seed(seed)

    temp = df.copy()

    known_locations = []

    for col in ce_cols + pe_cols:

        valid_rows = temp.index[
            temp[col].notna()
        ].tolist()

        for r in valid_rows:
            known_locations.append((r,col))

    known_locations = np.array(
        known_locations,
        dtype=object
    )

    n_hide = int(
        len(known_locations)
        * 0.20
    )

    selected_idx = np.random.choice(
        len(known_locations),
        n_hide,
        replace=False
    )

    hidden = []

    for idx in selected_idx:

        row,col = known_locations[idx]

        true_value = temp.loc[row,col]

        hidden.append(
            (row,col,true_value)
        )

        temp.loc[row,col] = np.nan

    # Fill CE
    for idx in temp.index:

        temp.loc[idx,ce_cols] = gap_aware_row(
            temp.loc[idx],
            ce_cols,
            ce_strikes
        )

        temp.loc[idx,pe_cols] = gap_aware_row(
            temp.loc[idx],
            pe_cols,
            pe_strikes
        )

    y_true = []
    y_pred = []

    for row,col,true_value in hidden:

        pred = temp.loc[row,col]

        # Safety fallback
        if pd.isna(pred):
            pred = np.nanmean(
                temp[col]
            )

        y_true.append(
            true_value
        )

        y_pred.append(
            pred
        )

    print(
        "NaNs in prediction:",
        np.isnan(y_pred).sum()
    )

    return mean_squared_error(
        y_true,
        y_pred
    )

In [11]:
def evaluate_gap_atm(seed=42):

    np.random.seed(seed)

    temp = df.copy()

    known_locations = []

    for col in ce_cols + pe_cols:

        valid_rows = temp.index[
            temp[col].notna()
        ].tolist()

        for r in valid_rows:
            known_locations.append((r,col))

    known_locations = np.array(
        known_locations,
        dtype=object
    )

    n_hide = int(
        len(known_locations)
        * 0.20
    )

    selected_idx = np.random.choice(
        len(known_locations),
        n_hide,
        replace=False
    )

    hidden = []

    for idx in selected_idx:

        row,col = known_locations[idx]

        true_value = temp.loc[row,col]

        hidden.append(
            (row,col,true_value)
        )

        temp.loc[row,col] = np.nan

    # Fill CE
    for idx in temp.index:

        temp.loc[idx,ce_cols] = gap_atm_row(
            temp.loc[idx],
            ce_cols,
            ce_strikes
        )

        temp.loc[idx,pe_cols] = gap_atm_row(
            temp.loc[idx],
            pe_cols,
            pe_strikes
        )

    y_true = []
    y_pred = []

    for row,col,true_value in hidden:

        pred = temp.loc[row,col]

        # Safety fallback
        if pd.isna(pred):
            pred = np.nanmean(
                temp[col]
            )

        y_true.append(
            true_value
        )

        y_pred.append(
            pred
        )

    print(
        "NaNs in prediction:",
        np.isnan(y_pred).sum()
    )

    return mean_squared_error(
        y_true,
        y_pred
    )

In [13]:
results = []

for seed in [1,2,3,4,5]:

    mse = evaluate_gap_aware(
        seed
    )

    results.append({
        "seed":seed,
        "gap_mse":mse
    })

results_df = pd.DataFrame(
    results
)

results_df

NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0


,seed,gap_mse
0,1,0.000102
1,2,0.000502
2,3,0.000974
3,4,0.000082
4,5,0.001463


In [14]:
results_df.mean(
    numeric_only=True
)

seed       3.000000
gap_mse    0.000625
dtype: float64

In [19]:
def gap_atm_row(row, cols, strikes):

    values = row[cols].astype(float).values.copy()

    # Full linear fill
    linear_result = linear_fill(
        values.copy(),
        strikes
    )

    # Full ATM-aware fill
    atm_result = atm_aware_row(
        row,
        cols,
        strikes
    )

    result = values.copy()

    n = len(values)

    i = 0

    while i < n:

        if np.isnan(values[i]):

            start = i

            while i < n and np.isnan(values[i]):
                i += 1

            end = i

            gap_size = end - start

            if gap_size == 1:

                result[start:end] = linear_result[
                    start:end
                ]

            else:

                result[start:end] = atm_result[
                    start:end
                ]

        else:

            i += 1

    return result

In [20]:
def evaluate_gap_atm(seed=42):

    np.random.seed(seed)

    temp = df.copy()

    known_locations = []

    for col in ce_cols + pe_cols:

        valid_rows = temp.index[
            temp[col].notna()
        ].tolist()

        for r in valid_rows:
            known_locations.append((r,col))

    known_locations = np.array(
        known_locations,
        dtype=object
    )

    n_hide = int(
        len(known_locations)
        * 0.20
    )

    selected_idx = np.random.choice(
        len(known_locations),
        n_hide,
        replace=False
    )

    hidden = []

    for idx in selected_idx:

        row,col = known_locations[idx]

        true_value = temp.loc[row,col]

        hidden.append(
            (row,col,true_value)
        )

        temp.loc[row,col] = np.nan

    for idx in temp.index:

        temp.loc[idx,ce_cols] = gap_atm_row(
            temp.loc[idx],
            ce_cols,
            ce_strikes
        )

        temp.loc[idx,pe_cols] = gap_atm_row(
            temp.loc[idx],
            pe_cols,
            pe_strikes
        )

    y_true = []
    y_pred = []

    for row,col,true_value in hidden:

        pred = temp.loc[row,col]

        if pd.isna(pred):
            pred = np.nanmean(temp[col])

        y_true.append(true_value)
        y_pred.append(pred)

    print(
        "NaNs in prediction:",
        np.isnan(y_pred).sum()
    )

    return mean_squared_error(
        y_true,
        y_pred
    )

In [21]:
results = []

for seed in [1,2,3,4,5]:

    mse = evaluate_gap_atm(seed)

    results.append({
        "seed":seed,
        "gap_atm_mse":mse
    })

results_df = pd.DataFrame(results)

print(results_df)
print()
print(results_df.mean(numeric_only=True))

NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0
NaNs in prediction: 0
   seed  gap_atm_mse
0     1     0.000067
1     2     0.000096
2     3     0.000137
3     4     0.000051
4     5     0.000111

seed           3.000000
gap_atm_mse    0.000092
dtype: float64


In [18]:
from scipy.interpolate import interp1d
import numpy as np

def atm_aware_row(row, cols, strikes):

    values = row[cols].astype(float).values.copy()

    spot = row["underlying_price"]

    atm_idx = np.argmin(
        np.abs(strikes - spot)
    )

    result = values.copy()

    # LEFT SIDE
    left_strikes = strikes[:atm_idx+1]
    left_values = values[:atm_idx+1]

    mask = ~np.isnan(left_values)

    if mask.sum() >= 2:

        f = interp1d(
            left_strikes[mask],
            left_values[mask],
            kind="linear",
            fill_value="extrapolate",
            bounds_error=False
        )

        missing = np.isnan(left_values)

        result[:atm_idx+1][missing] = f(
            left_strikes[missing]
        )

    # RIGHT SIDE
    right_strikes = strikes[atm_idx:]
    right_values = values[atm_idx:]

    mask = ~np.isnan(right_values)

    if mask.sum() >= 2:

        f = interp1d(
            right_strikes[mask],
            right_values[mask],
            kind="linear",
            fill_value="extrapolate",
            bounds_error=False
        )

        missing = np.isnan(right_values)

        result[atm_idx:][missing] = f(
            right_strikes[missing]
        )

    # fallback
    mask = ~np.isnan(values)

    if np.isnan(result).any() and mask.sum() >= 2:

        f = interp1d(
            strikes[mask],
            values[mask],
            kind="linear",
            fill_value="extrapolate",
            bounds_error=False
        )

        missing = np.isnan(result)

        result[missing] = f(
            strikes[missing]
        )

    return result